# ME 3300 Prelab 03 Walkthrough — Automation & Derivatives

Lab 03 brings your first **DAQ automation** (scripting the ADS with `pydwf`)
and your first **numerical derivative**. This walkthrough teaches the new
Python patterns on simulated signals — no hardware needed. Real acquisition
happens in lab.

Run cell by cell; report **CHECKPOINT** values in the Prelab 03 quiz on
Canvas. Select your `.venv` kernel first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # seeded: everyone gets the same numbers
print('imports OK')

## 1. `while` loops — wait until a condition

A `for` loop runs a *known* number of times. A **`while` loop** runs until a
condition changes — which is exactly what acquisition code needs: "keep
waiting *until* the hardware says Done." `break` exits the loop immediately.

In [ ]:
# Simulated hardware: a fake status counter that finishes on the 5th poll
poll_count = 0

def fake_status():
    global poll_count
    poll_count += 1
    return 'Done' if poll_count >= 5 else 'Busy'

while True:
    status = fake_status()
    print(f"poll {poll_count}: {status}")
    if status == 'Done':
        break

print(f"acquisition complete after {poll_count} polls")

This is precisely the shape of the pydwf pattern you'll use in lab:

```python
while True:
    status = ai.status(True)
    if status == DwfState.Done:
        break
    time.sleep(0.01)
```

**CHECKPOINT 1:** Change the fake hardware to finish on the **8th** poll.
How many lines does the loop print now?

## 2. `input()` — pausing for the human

`input('message')` prints the message and pauses until you press Enter
(returning whatever you typed). Acquisition scripts use it to wait while you
physically position the apparatus. Try it — run this cell, type your name in
the box that appears (top of the VS Code window), press Enter:

In [ ]:
name = input('Type your name and press Enter: ')
print(f'Hello, {name}! The script waited for you.')

## 3. Lists, `.append`, and `zip`

In Lab 02 you pre-allocated arrays with `np.zeros(...)` and filled slots by
index. When results simply arrive one at a time, a plain **list** you
`.append` to is more natural:

In [ ]:
measurements = []            # empty list
for trial in range(4):
    fake_reading = 1.65 + 0.1*trial
    measurements.append(fake_reading)

print(measurements)
measurements = np.array(measurements)   # convert for math
print('mean:', measurements.mean())

In [ ]:
# zip: loop over two lists IN PARALLEL, one matched pair per pass
positions   = ['right', 'up', 'left', 'down']
known_accel = [0.0, 9.81, 0.0, -9.81]

for pos, accel in zip(positions, known_accel):
    print(f"at '{pos}' the sensor should read {accel:+.2f} m/s^2")

Compare with `enumerate` (Lab 02): `enumerate` pairs each element with
its *index*; `zip` pairs elements of one list with elements of *another*.

**CHECKPOINT 2:** Using `zip`, loop over `[1, 2, 3]` and `[10, 20, 30]` and
append the *products* to a list. What is the sum of that list?

## 4. `np.column_stack` — saving multi-column results

In [ ]:
t = np.arange(0, 1, 0.25)
x = t**2
table = np.column_stack([t, x])      # two columns, side by side
print(table)
np.savetxt('practice_table.csv', table, header='t_s,x_m', delimiter=',')

# ...and np.loadtxt reads it back, skipping the '#' header automatically:
back = np.loadtxt('practice_table.csv', delimiter=',', comments='#')
print('round trip OK:', np.allclose(table, back))

Note what `np.savetxt` did with `header=`: it wrote the line prefixed
with `#`. That's why `comments='#'` on the way back in skips it — the same
convention your saved calibration files use.

## 5. `np.gradient` — the derivative, done right

Lab 02's `np.diff` gives differences between neighbors: N points in,
**N−1** out, each value landing *between* two samples. For a derivative you
plot against the original time axis, that off-by-one-and-a-half-shift is a
nuisance. **`np.gradient`** uses *central* differences (each slope computed
from both neighbors) and returns the **same length** as its input:

In [ ]:
dt = 0.01
t_sig = np.arange(0, 2, dt)
x_sig = np.sin(2*np.pi*t_sig)               # position
v_true = 2*np.pi*np.cos(2*np.pi*t_sig)      # exact derivative

v_num = np.gradient(x_sig, dt)              # numerical derivative

print(f"len(x) = {len(x_sig)}, len(gradient) = {len(v_num)}, "
      f"len(diff) = {len(np.diff(x_sig))}")
print(f"max error vs exact: {np.abs(v_num - v_true).max():.4f}")

**CHECKPOINT 3:** What is `len(np.gradient(x))` for an array `x` of
length 200? What is `len(np.diff(x))`?

### Why derivatives amplify noise

Now the important physics-of-measurement lesson. Add a little noise to the
position signal and differentiate again:

In [ ]:
x_noisy = x_sig + rng.normal(0, 0.001, x_sig.size)   # tiny 0.001 noise!
v_noisy = np.gradient(x_noisy, dt)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.5, 5), sharex=True)
ax1.plot(t_sig, x_noisy, 'b-', linewidth=1)
ax1.set_ylabel('position'); ax1.set_title('signal: noise is invisible')
ax1.grid(True)
ax2.plot(t_sig, v_noisy, 'b-', linewidth=0.8, label='gradient of noisy signal')
ax2.plot(t_sig, v_true, 'r-', linewidth=2, label='true derivative')
ax2.set_ylabel('velocity'); ax2.set_xlabel('time (s)')
ax2.set_title('derivative: the same noise, amplified')
ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

noise_amplification = v_noisy.std() / v_true.std()
print(f"derivative noise is clearly visible even though position noise "
      f"was only 0.001")

The noise was invisible in the position trace but dominates the
derivative. Differentiation divides each sample-to-sample noise wiggle by
the tiny `dt`, magnifying it enormously. **This is why your calculated
normal acceleration in lab will be much noisier than the accelerometer's
direct measurement** — and it's why we still record fast (the dense samples
let the derivative trace the real curve; the noise cost is unavoidable
either way).

**CHECKPOINT 4:** Re-run the cell above with noise `0.005` instead of
`0.001`. Roughly what is the peak spurious velocity (largest wrong value)
in the derivative plot? Report to the nearest whole number.

## 6. Preview: the pydwf pattern

You can't run this at home without an ADS, but read it now so it's familiar
in lab (do NOT run this cell unless hardware is attached):

```python
from pydwf import DwfLibrary, DwfState
from pydwf.utilities import openDwfDevice
import time

dwf = DwfLibrary()
with openDwfDevice(dwf) as device:     # auto-closes, even on errors
    ai = device.analogIn               # the Scope, as an object
    ai.channelEnableSet(0, True)       # GUI 'Channel 1' = index 0 !
    ai.frequencySet(1000)              # = typing 1000 Hz in the GUI
    ai.bufferSizeSet(2000)
    ai.configure(False, True)          # start acquisition
    while ai.status(True) != DwfState.Done:
        time.sleep(0.01)               # the polling loop from section 1
    volts = np.array(ai.statusData(0, 2000))
```

Every line maps to a GUI action you performed in Lab 02. The one trap:
**pydwf counts channels from 0**, so GUI Channel 2 is index 1 in code.

**CHECKPOINT 5:** In the code above, what GUI channel does
`ai.statusData(0, 2000)` read from? What would you change to read GUI
Channel 2?

## Done!

| Skill | Where you'll use it in lab |
|---|---|
| `while` + `break` polling | waiting for each pydwf acquisition |
| `input()` | pausing between the 4 calibration orientations |
| list + `.append`, `zip` | collecting the 4 calibration means |
| `np.column_stack` / `loadtxt(comments='#')` | saving/loading calibrations |
| `np.gradient` | angular velocity from the angle signal |
| noise-amplification insight | explaining your comparison plot |

Report the checkpoints in the **Prelab 03 quiz on Canvas**.